# Qwen3-4B medical fine-tune (evaluation)

Scores the fine-tuned adapter against the base model on two benchmarks it never
trained on: MedQA-USMLE test (1,273) and MedMCQA validation (4,183). Same
prompts, same greedy decoding, both models from one load -- the base is the same
weights with the adapter switched off.

It runs in two stages in one session. A smoke pass scores 32 questions per
benchmark through the exact code the full run uses, measures its speed, and
stops if the full run would not fit the session. Only then does the full run
start.

**Sidebar: Accelerator `GPU T4 x2`, Internet `On`.** Attach `medical-ft-code`
and `medical-ft-adapter`.

In [ ]:
# --- 1. Hardware check (stops here if the GPU is unusable) -----------------
import os, subprocess

# Two T4s are offered, but a 4B model in 4-bit is ~3.3GB against 15.6GB of card.
# Splitting it buys nothing and pays PCIe on every forward and backward. Pin to
# one GPU BEFORE torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

name = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                       "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
major, minor = torch.cuda.get_device_capability()
print(f"GPU         : {name}")
print(f"visible GPUs: {torch.cuda.device_count()} (pinned to one on purpose)")
print(f"capability  : {major}.{minor}")
print(f"torch       : {torch.__version__}   bf16: {torch.cuda.is_bf16_supported()}")

if major < 7:
    raise SystemExit(
        f"\nSTOP. Compute capability {major}.{minor} ({name.split(',')[0]}) has no "
        "kernels in modern PyTorch builds.\nFIX: sidebar -> Accelerator -> "
        "'GPU T4 x2', then Run All. The GPU type cannot be set through the API.")
print("\nGPU supported." if torch.cuda.is_bf16_supported()
      else "\nGPU supported. Turing has no bf16; fp16 is selected automatically.")

In [ ]:
%%capture
# --no-deps keeps pip from replacing the preinstalled transformers 5.5.0 this code was checked against.
!pip install -q --no-deps peft
!pip install -q datasets

In [ ]:
# --- 2. Verify the install before spending GPU time on it ------------------
import torch, transformers, peft
print(f"ok | torch {torch.__version__} | transformers {transformers.__version__} "
      f"| peft {peft.__version__}")

In [ ]:
# --- 3. Get the code and the adapter from the attached datasets ------------
import os, shlex, shutil, subprocess, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK  = Path("/kaggle/working/ft")
PKG   = WORK / "training"
PKG.mkdir(parents=True, exist_ok=True)

REQUIRED = {"records", "prompts", "sources", "prepare_data",
            "check_lengths", "budget", "train"}

EVAL_REQUIRED = {"records", "prompts", "sources", "evalcore", "evaluate",
                 "modeling"}


def find_code_dir(root: Path, required: set[str] = REQUIRED) -> Path:
    """Locate the uploaded modules wherever Kaggle mounted them.

    The mount path is not stable: a dataset declared as gb1105/medical-ft-code
    turned up under /kaggle/input/datasets/... rather than at
    /kaggle/input/medical-ft-code. Two runs died on that assumption, so this
    searches for the directory that actually holds the modules instead.
    """
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-code.")
    candidates = []
    for path in root.rglob("*.py"):
        stems = {p.stem for p in path.parent.glob("*.py")}
        if required <= stems:
            candidates.append(path.parent)
    if not candidates:
        found = sorted(str(p.relative_to(root)) for p in root.rglob("*.py"))[:20]
        tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
        raise SystemExit(
            f"\nSTOP. No directory under {root} contains all of {sorted(required)}.\n"
            f".py files found: {found or 'none'}\n"
            f"First entries under /kaggle/input: {tree}\n"
            "FIX: re-run scripts/push_kaggle.sh, then confirm the "
            "medical-ft-code dataset is attached in the sidebar.")
    return sorted(set(candidates))[0]


def find_adapter_dir(root: Path) -> Path:
    """Locate the uploaded LoRA adapter by content, the same way.

    A directory qualifies when adapter_config.json and adapter_model.safetensors
    sit side by side. A final adapter wins over any checkpoint-* directory, so a
    stray checkpoint cannot be scored in place of the finished run.
    """
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-adapter.")
    found = sorted({p.parent for p in root.rglob("adapter_config.json")
                    if (p.parent / "adapter_model.safetensors").exists()})
    final = [d for d in found if not d.name.startswith("checkpoint-")]
    if final or found:
        return (final or found)[0]
    tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
    raise SystemExit(
        f"\nSTOP. No LoRA adapter under {root}: no adapter_config.json with "
        "adapter_model.safetensors beside it.\n"
        f"First entries under /kaggle/input: {tree}\n"
        "FIX: run scripts/push_adapter.sh, then attach medical-ft-adapter in "
        "the sidebar.")


def code_fingerprint(directory: Path) -> str:
    """A short hash of every .py file's name and bytes, in name order.

    Each notebook is built for one exact set of modules. If Kaggle mounts an
    older version of the code dataset -- a new version still processing, or an
    old one attached by hand -- the modules carry the right names and the wrong
    code, and nothing else would notice.
    """
    import hashlib

    digest = hashlib.sha256()
    for path in sorted(directory.glob("*.py")):
        digest.update(path.name.encode() + b"\0" + path.read_bytes() + b"\0")
    return digest.hexdigest()[:16]

SRC = find_code_dir(INPUT, EVAL_REQUIRED)
EXPECTED_FINGERPRINT = "9f55d00ae45b2f5f"
if code_fingerprint(SRC) != EXPECTED_FINGERPRINT:
    raise SystemExit(
        f"\nSTOP. The attached code ({code_fingerprint(SRC)}) is not the code this "
        f"notebook was built for ({EXPECTED_FINGERPRINT}).\n"
        "Kaggle may still be processing a new version of medical-ft-code, or an "
        "older version is attached.\n"
        "FIX: wait a minute and re-run; if it persists, run scripts/push_kaggle.sh again.")
ADAPTER = find_adapter_dir(INPUT)
# step() runs through a shell, so the path is quoted: a mount path with a space
# in it would otherwise split into two arguments.
ADAPTER_ARG = shlex.quote(str(ADAPTER))
print("code:   ", SRC)
print("adapter:", ADAPTER)

for src_file in sorted(SRC.glob("*.py")):
    shutil.copy(src_file, PKG / src_file.name)
(PKG / "__init__.py").touch()

os.chdir(WORK)
sys.path.insert(0, str(WORK))
Path("data").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

# A failing `!python x.py` returns non-zero but does not raise in Jupyter, so the
# notebook would sail past a dead step and fail later somewhere confusing.
def step(cmd: str):
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise SystemExit(f"\nStep failed (exit {p.returncode}):\n  {cmd}")
    print("\nok\n", flush=True)

## 4. Build the held-out sets

Exactly the sets training was decontaminated against. MedMCQA validation is
loaded with both filters off: training drops rows without an explanation and
rows marked multi-choice, but the benchmark keeps all 4,183, or the score would
not be comparable to any published MedMCQA figure.

In [ ]:
import json
from training.evaluate import EXPECTED_HOLDOUT_SIZES as EXPECTED
from training.sources import load

HOLDOUTS = {
    "medqa": ("test", {}),
    "medmcqa": ("validation", {"require_rationale": False,
                               "require_single_choice": False}),
}

for name, (split, flags) in HOLDOUTS.items():
    recs = load(name, limit=0, split=split, **flags)
    if len(recs) != EXPECTED[name]:
        raise SystemExit(
            f"\nSTOP. {name} {split} gave {len(recs):,} rows, expected "
            f"{EXPECTED[name]:,}.\n"
            "FIX: the dataset on the Hugging Face Hub has changed since these "
            "sizes were confirmed. Check its revision before scoring anything.")
    with open(f"data/holdout_{name}.jsonl", "w") as fh:
        for rec in recs:
            fh.write(json.dumps(rec.to_dict()) + "\n")
    print(f"data/holdout_{name}.jsonl  {len(recs):,}")

## 5. Smoke pass

32 questions per benchmark through the exact code the full run uses. It proves
the path works on this GPU, measures real throughput, and stops here if the full
run would not fit the session. It also stops if the fine-tune's answers mostly
cannot be parsed, which would mean truncated output rather than a real
result.

In [ ]:
from training.evaluate import project_eval_seconds

SESSION_BUDGET = 27_000      # 7.5h of Kaggle's 9h GPU session; the rest is margin
GEN_LIMIT = 300

projected = 0
for name in ("medqa", "medmcqa"):
    step(f"python -m training.evaluate --adapter {ADAPTER_ARG} "
         f"--test data/holdout_{name}.jsonl --limit 32 --gen-limit 16 "
         f"--out outputs/smoke_{name}.json")
    smoke = json.load(open(f"outputs/smoke_{name}.json"))
    projected += project_eval_seconds(smoke["timing"],
                                      n_constrained=EXPECTED[name],
                                      n_generative=GEN_LIMIT)
    gen = smoke["reports"]["generative"]
    print(f"{name}: tuned answers unparseable {gen['tuned_unparseable']}/{gen['n']}, "
          f"base {gen['base_unparseable']}/{gen['n']}")
    if gen["tuned_unparseable"] > gen["n"] // 2:
        raise SystemExit(
            f"\nSTOP. {gen['tuned_unparseable']} of {gen['n']} fine-tuned answers "
            "could not be parsed, which means truncated output, not a result.\n"
            "FIX: raise --max-new-tokens in the full run.")

print(f"\nprojected full run: {projected / 3600:.2f}h "
      f"against a {SESSION_BUDGET / 3600:.1f}h budget")
if projected > SESSION_BUDGET:
    raise SystemExit(
        f"\nSTOP. The full evaluation projects to {projected / 3600:.1f}h.\n"
        "FIX: lower GEN_LIMIT; generation dominates the runtime.")
print("fits; starting the full run")

## 6. Full evaluation

All 1,273 and all 4,183 questions by constrained scoring, and the first 300 of
each by greedy generation, paired against the base model.

In [ ]:
for name in ("medqa", "medmcqa"):
    step(f"python -m training.evaluate --adapter {ADAPTER_ARG} "
         f"--test data/holdout_{name}.jsonl --gen-limit {GEN_LIMIT} "
         f"--out outputs/eval_{name}.json")

In [ ]:
# --- 7. Results, and save everything to the Output panel -------------------
out = Path("/kaggle/working")
for label, name in (("MedQA-USMLE test", "medqa"),
                    ("MedMCQA validation", "medmcqa")):
    data = json.load(open(f"outputs/eval_{name}.json"))
    print(f"\n########## {label} ##########")
    for mode, rep in data["reports"].items():
        m = rep["mcnemar"]
        delta = (rep["tuned_accuracy"] - rep["base_accuracy"]) * 100
        print(f"  {mode:<12} n={rep['n']:>5,}  base {rep['base_accuracy']:6.1%}  "
              f"tuned {rep['tuned_accuracy']:6.1%}  ({delta:+.1f})  "
              f"wins {m['wins']} / regressions {m['regressions']}  "
              f"p={m['p_value']:.4f}")
        if m["p_value"] >= 0.05:
            print("               not significant at p<0.05: "
                  "indistinguishable from base")
    shutil.copy(f"outputs/eval_{name}.json", out / f"eval_{name}.json")
print("\nSaved eval_medqa.json and eval_medmcqa.json to the Output panel.")

## Done

`eval_medqa.json` and `eval_medmcqa.json` are in the **Output** panel: accuracy
for both models in both modes, McNemar significance, a per-subject breakdown,
timing, and twelve raw completions per model for reading by eye.